In [5]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.tools import tool
import requests
from langchain_community.tools import DuckDuckGoSearchRun
from langchain.agents import create_agent
from langsmith import Client
import os
from dotenv import load_dotenv

In [7]:
load_dotenv()

True

In [2]:
search_tool = DuckDuckGoSearchRun()

In [ ]:
# GETTING API KEY
API_KEY = os.getenv("WEATHERSTACK_API_KEY")

# tool create
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key={API_KEY}&query={city}'

  response = requests.get(url)

  return response.json()

In [11]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [12]:
client = Client()
prompt = client.pull_prompt("hwchase17/react")

In [13]:
agent = create_agent(
    model=llm,
    tools=[search_tool, get_weather_data],
    system_prompt=prompt.template,
    debug=True
)

In [14]:
response = agent.invoke(
    {"messages": [{"role": "user", "content": "What is the capital of Himachal Pradesh then find it's current weather condition"}]}
)

[values] {'messages': [HumanMessage(content="What is the capital of Himachal Pradesh then find it's current weather condition", additional_kwargs={}, response_metadata={}, id='ac6dd8f6-fe4e-4f41-9aa4-a9a9c627029f')]}
[updates] {'model': {'messages': [AIMessage(content='', additional_kwargs={'function_call': {'name': 'duckduckgo_search', 'arguments': '{"query": "capital of Himachal Pradesh"}'}, '__gemini_function_call_thought_signatures__': {'60e6e767-e77e-403e-8dd1-9d139dd0a61e': 'CqgCAb4+9vugsbXb4P26jslo97VRYs/KRhWpGOcKGixAtMMwANFK2PfVakzfuB8CBFMakzdP2Y1Bsy3Yar1x4WV7bwcXkDSiQtV0PmeDtYAYFfA4h/P1/yZDI8WT9L10fxjunnZs4GAkfqLOShzlYm/YG0f8W0PqVLGX+hqF64VRoRjjO0qM18ld3zADhFZKsOe9fLOi7vh/49J8ZEApz0HafK0ypl9viW/rHjCxl44BLZ1Nuw2chfJ25or3h4IhsV9lDN8V7LsHYFGAWwICGw7BeH/Gp5psiWgZpxVqQhlnbM+E0Lt2phj+BdnNSDqTw3XMxxlgGrcVXconhaYZaEWtGMm9MDI4g2hJaF9sx+BbEwbtiuGQEsqz+8HA2iQVOWPbqGEdZeZAzUI='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_pr

In [17]:
content = response["messages"][-1].content

# If it's a list (Gemini style)
if isinstance(content, list):
    content = content[0]["text"]

print(content)

The capital of Himachal Pradesh is Shimla (summer capital) and Dharamshala (winter capital). The current weather in Shimla is clear, with a temperature of 15 degrees Celsius, a feels like temperature of 15 degrees Celsius, 33% humidity, and a wind speed of 6 km/h from the NW direction.
